In [ ]:
import sys

# Install SHAP into the *current* Jupyter kernel environment.
# If installation succeeds but import still fails, restart the kernel and re-run.
!"{sys.executable}" -m pip install -q --upgrade pip
!"{sys.executable}" -m pip install -q shap matplotlib
print("Python:", sys.executable)

In [ ]:
from pathlib import Path
import json

import joblib
import numpy as np
import matplotlib.pyplot as plt
import shap

In [ ]:
SEED = 42
NSAMPLES = 5000  # SHAP sampling budget per instance (increase for more stability)
K_TOP = 10       # for compact display/saving
BACKGROUND_SIZE = 50

DATA_DIR = Path("data_processed")
MLP_DIR = Path("artifacts/mlp")
LIME_DIR = Path("artifacts/lime")
OUT_DIR = Path("artifacts/shap")
OUT_DIR.mkdir(parents=True, exist_ok=True)

X_train = np.load(DATA_DIR / "X_train.npy")
y_train = np.load(DATA_DIR / "y_train.npy")
X_test = np.load(DATA_DIR / "X_test.npy")
y_test = np.load(DATA_DIR / "y_test.npy")

meta = json.loads((DATA_DIR / "metadata.json").read_text(encoding="utf-8"))
feature_names = meta["feature_names"]
class_names = meta["target_names"]

model = joblib.load(MLP_DIR / "model.joblib")

chosen = json.loads((LIME_DIR / "chosen_test_indices.json").read_text(encoding="utf-8"))
test_indices = [int(i) for i in chosen["chosen_test_indices"]]
len(test_indices), test_indices[:5]

In [ ]:
(OUT_DIR / "chosen_test_indices.json").write_text(
    json.dumps(chosen, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
(OUT_DIR / "chosen_test_indices.json").as_posix()

In [ ]:
rng = np.random.default_rng(SEED)
bg_idx = rng.choice(np.arange(X_train.shape[0]), size=min(BACKGROUND_SIZE, X_train.shape[0]), replace=False)
bg_idx = [int(i) for i in bg_idx]
background = X_train[bg_idx]

(OUT_DIR / "background_indices.json").write_text(
    json.dumps({"seed": SEED, "background_size": int(len(bg_idx)), "indices": bg_idx}, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
background.shape

In [ ]:
def predict_proba_class1(X):
    # Explain a single scalar output for stability: P(class=1) ("benign")
    return model.predict_proba(X)[:, 1]

# KernelExplainer for model-agnostic SHAP (scalar output)
explainer = shap.KernelExplainer(predict_proba_class1, background)
explainer.expected_value

In [ ]:
(OUT_DIR / "expected_value.json").write_text(
    json.dumps({"expected_value": float(explainer.expected_value)}, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
(OUT_DIR / "expected_value.json").as_posix()

In [ ]:
shap_class1 = []
per_instance = []

for idx in test_indices:
    x = X_test[idx]
    true_y = int(y_test[idx])
    proba = model.predict_proba(x.reshape(1, -1))[0]
    pred_y = int(np.argmax(proba))

    # Make sampling deterministic
    np.random.seed(SEED + int(idx))

    x2d = x.reshape(1, -1)
    sv = explainer.shap_values(x2d, nsamples=NSAMPLES)
    # sv is shaped (1, n_features)
    sv1 = np.array(sv).reshape(-1)
    shap_class1.append(sv1)

    # Compact top-K by absolute value for JSON summary
    top = np.argsort(np.abs(sv1))[-K_TOP:][::-1]
    weights_top = [
        {"feature": feature_names[int(j)], "value": float(x[int(j)]), "shap": float(sv1[int(j)])}
        for j in top
    ]

    per_instance.append(
        {
            "test_index": int(idx),
            "true_y": true_y,
            "pred_y": pred_y,
            "proba": [float(p) for p in proba],
            "topK_class1": weights_top,
        }
    )

    # Waterfall plot for class 1 probability
    try:
        exp_obj = shap.Explanation(
            values=sv1,
            base_values=float(explainer.expected_value),
            data=x,
            feature_names=feature_names,
        )
        shap.plots.waterfall(exp_obj, max_display=K_TOP, show=False)
        plt.tight_layout()
        png_path = OUT_DIR / f"waterfall_test_{idx:03d}_class1.png"
        plt.savefig(png_path, dpi=150)
        plt.close()
    except Exception as e:
        per_instance[-1]["plot_error"] = str(e)

shap_class1 = np.stack(shap_class1, axis=0)

np.save(OUT_DIR / "shap_values_class1.npy", shap_class1)

summary = {
    "seed": SEED,
    "nsamples": int(NSAMPLES),
    "k_top": int(K_TOP),
    "background_size": int(len(bg_idx)),
    "n_examples": int(len(test_indices)),
    "class_names": class_names,
    "results": per_instance,
}
(OUT_DIR / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")

shap_class1.shape